<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week7_debugging_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before running your model: predict the three most likely failure modes. Overfitting? Class imbalance? Out-of-vocabulary terms? Write your predictions, then diagnose after training.

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 7 — When the Model Is Wrong (STARTER)
### The NLP Internship | LinguaAI Labs

This week: diagnose 'account' domain failures on CLINC150, augment, and implement active learning.

**Write your hypothesis before any code.**

## My Hypothesis

**Primary cause of 'account' misclassification:** [WRITE HERE — short query / lexical overlap with billing / class imbalance]
**Reasoning:** [WRITE HERE]

In [ ]:
import sys, subprocess, os, random
for pkg in ["datasets","scikit-learn","nltk"]:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
from datasets import load_dataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
import nltk, warnings; warnings.filterwarnings("ignore")
for r in ["wordnet","averaged_perceptron_tagger","punkt","punkt_tab"]:
    nltk.download(r, quiet=True)
np.random.seed(42); random.seed(42); os.makedirs("outputs", exist_ok=True)

DOMAIN_MAP = {
    "bill_balance":"billing","bill_pay":"billing","transfer":"billing","pay_bill":"billing",
    "min_payment":"billing","account_blocked":"billing",
    "freeze_account":"account","lost_stolen_card":"account","report_fraud":"account",
    "card_declined":"account","cancel_card":"account","credit_score":"account",
    "travel_alert":"travel","flight_status":"travel","book_hotel":"travel","book_flight":"travel",
    "reminder":"utility","calendar":"utility","schedule_meeting":"utility","make_call":"utility",
}
clinc = load_dataset("clinc_oos", "plus")
label_names = clinc["train"].features["intent"].names
def collapse(i): return DOMAIN_MAP.get(label_names[i], "other")
df_train = clinc["train"].to_pandas().rename(columns={"text":"text_clean"})
df_test  = clinc["test"].to_pandas().rename(columns={"text":"text_clean"})
df_train["category"] = df_train["intent"].apply(collapse)
df_test["category"]  = df_test["intent"].apply(collapse)
X_train=df_train["text_clean"].values; y_train=df_train["category"].values
X_test =df_test["text_clean"].values;  y_test =df_test["category"].values
tfidf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)),
    ("clf",   LogisticRegression(multi_class="multinomial", max_iter=1000,
                                  class_weight="balanced", random_state=42)),
])
tfidf_pipeline.fit(X_train, y_train)
y_pred_baseline = tfidf_pipeline.predict(X_test)
baseline_f1 = f1_score(y_test, y_pred_baseline, average="weighted", zero_division=0)
print(f"CLINC150 loaded ✅  Baseline F1={baseline_f1:.3f}")

## Part 1 — Error Analysis

> ⏸️ **Predict:** which domain is 'account' most confused with?

In [ ]:
error_df = pd.DataFrame({"text":X_test,"true":y_test,"pred":y_pred_baseline})
account_errors = error_df[(error_df["true"]=="account")&(error_df["pred"]!="account")]
print(f"Account errors: {len(account_errors)}")
print(account_errors["pred"].value_counts())
for _, row in account_errors.head(15).iterrows():
    print(f"  [→{row['pred']}] {row['text'][:90]}")

In [ ]:
error_taxonomy = {
    "lexical_overlap_billing": None,  # YOUR CODE HERE
    "short_query_ambiguity":   None,  # YOUR CODE HERE
    "genuine_boundary":        None,  # YOUR CODE HERE
    "labelling_inconsistency": None,  # YOUR CODE HERE
    "sarcasm_adversarial":     None,  # YOUR CODE HERE
}
total = sum(v for v in error_taxonomy.values() if v is not None)
if total > 0:
    for k,v in error_taxonomy.items():
        if v is not None: print(f"  {k:<32}: {v} ({v/total:.0%})")
else:
    print("Fill in error_taxonomy counts above.")

## Part 2 — Class Imbalance

In [ ]:
from collections import Counter
train_counts = Counter(y_train)
for cat, count in sorted(train_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 40)
    print(f"  {cat:<22} {count:>4} ({count/len(y_train):.1%})  {bar}")
account_n = None  # YOUR CODE HERE
majority  = None  # YOUR CODE HERE
ratio     = None  # YOUR CODE HERE
print(f"\nImbalance ratio: {ratio}")

## Part 3 — EDA Augmentation

In [ ]:
from nltk.corpus import wordnet

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            if lemma.name() != word and "_" not in lemma.name():
                synonyms.add(lemma.name())
    return list(synonyms)

def synonym_replacement(text, n=2):
    # YOUR CODE HERE
    pass

def random_deletion(text, p=0.1):
    # YOUR CODE HERE
    pass

sample = "I need to freeze my credit card immediately it was stolen"
print("Original: ", sample)
print("SR aug 1: ", synonym_replacement(sample))
print("SR aug 2: ", synonym_replacement(sample))
print("RD aug:   ", random_deletion(sample, 0.1))

In [ ]:
account_train = [X_train[i] for i in range(len(y_train)) if y_train[i]=="account"]
aug_texts, aug_labels = [], []
for text in account_train:
    # YOUR CODE HERE — original + 2 SR + 1 RD
    pass

X_train_aug = list(X_train) + aug_texts
y_train_aug = list(y_train) + aug_labels
aug_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)),
    ("clf",   LogisticRegression(multi_class="multinomial", max_iter=1000,
                                  class_weight="balanced", random_state=42)),
])
aug_pipeline.fit(X_train_aug, y_train_aug)
y_pred_aug = aug_pipeline.predict(X_test)
aug_f1 = f1_score(y_test, y_pred_aug, average="weighted", zero_division=0)
print(f"Before: {baseline_f1:.3f} → After augmentation: {aug_f1:.3f}")
for name_, y_pred in [("Baseline",y_pred_baseline),("Augmented",y_pred_aug)]:
    rep = classification_report(y_test,y_pred,output_dict=True,zero_division=0)
    print(f"  {name_:<12}: account recall={rep.get('account',{}).get('recall',0):.3f}")

## Part 4 — Sarcasm and Adversarial Inputs

In [ ]:
sarcastic = [
    "Oh great, my card got declined AGAIN. Fantastic service.",
    "Truly impressed — my account locked for a week and nobody noticed.",
]
print("Predict BEFORE running:\n")
for q in sarcastic:
    pred = None  # YOUR CODE HERE — aug_pipeline.predict([q])[0]
    print(f"  [→{pred}] {q[:80]}")

## Part 5 — Active Learning: Uncertainty Sampling

In [ ]:
sample_texts = df_train["text_clean"].values[:2000]
proba         = None  # YOUR CODE HERE — aug_pipeline.predict_proba(sample_texts)
uncertainty   = None  # YOUR CODE HERE — 1 - proba.max(axis=1)
top_uncertain = None  # YOUR CODE HERE — np.argsort(uncertainty)[::-1]

for rank, idx in enumerate(top_uncertain[:10], 1):
    cats = aug_pipeline.classes_; top2 = proba[idx].argsort()[::-1][:2]
    print(f"  [{rank}] {uncertainty[idx]:.3f} | {sample_texts[idx][:70]}")
    print(f"       {cats[top2[0]]}={proba[idx][top2[0]]:.2f} | {cats[top2[1]]}={proba[idx][top2[1]]:.2f}")

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Treating a high training F1 as success | A classifier scoring 0.95 on train and 0.61 on test has memorised training examples, not learned the pattern. | Always compare train and validation metrics side by side |
| Debugging only with aggregate metrics | Overall F1 hides class-level failures. Your model might work perfectly for billing but fail completely for technical issues. | Always plot the full confusion matrix and per-class report before declaring the model fixed |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Document three things you tried in this notebook that did not improve the model. For each, write one sentence explaining why it did not work.

> 🔬 **Advanced Path**
> Plot learning curves. Diagnose the failure mode. Implement one fix (regularisation, resampling, or feature engineering) and show before/after metrics.

## ✅ What You Can Do After This Week

- Diagnose CLINC150 errors into a taxonomy
- Augment the minority domain and measure recall improvement
- Identify sarcasm as a real failure mode on short-text data
- Implement uncertainty sampling for efficient labelling

---
## ✅ Week 7 Complete — Next: `week8_transformers_STARTER.ipynb`

---
*Add `week7_debugging_STARTER.ipynb` to your portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 7, you can now:

- Diagnose model failure using learning curves, confusion matrices, and per-class metrics
- Distinguish between overfitting, underfitting, and distribution mismatch
- Document negative results — what you tried and why it did not work
- Apply one targeted fix and measure its impact rigorously

📤 **GitHub:** Push week7_debugging_STARTER.ipynb and debugging_log.md. Commit: "Week 7: Debugging — failure mode documented"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
